# Phase 4 — 교차모델 behavior 주입 (성격 착시 재현) · v2 (Qwen 수리)

**질문:** v_behavior를 mid-layer에 주입하면 **행동은 움직이나 자기보고 digit은 3에 고정**되는 성격 착시가 Mistral·Qwen에도 나타나나?

**v2 변경(1차서 Qwen 무효 → 수리):**
- **Qwen `--standardize`**: 차원별 z-score로 거대활성(massive activation) 다운웨이트 → v_behavior 잡음화 방지.
- **강도 축소** `coeffs=-1..1`(±2 과주입 제거, Qwen 생성붕괴 방지).
- **진단 셀**: extract 후 `diagnostics.py`로 `cos_V1_V2`·거대활성 `top1%`를 층별 확인.
- **비교표 `cos_V_cue` 게이트**: 벡터가 외향축을 잡았는지(<0.1이면 "벡터불량"). dE만 보고 속지 않게.

**파이프라인(모델별 순차):** `extract → diagnostics → build(v_behavior) → steer_eval(α-sweep)`.
**번들:** 코드 수정 반영된 `steering_crossmodel_bundle.zip` 재빌드본 업로드. **-it/Instruct 전용.**

## 1. GPU + 의존성 + HF 로그인

In [ ]:
!nvidia-smi -L

In [ ]:
!pip install -q "transformers>=4.45" accelerate huggingface_hub bitsandbytes sentencepiece python-dotenv
import torch; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from huggingface_hub import notebook_login
notebook_login()   # gemma-2-9b-it 라이선스 수락 필요. Mistral/Qwen(Apache)은 로그인 없어도 됨.

## 2. 번들 업로드

`steering_crossmodel_bundle.zip`(재빌드본 — build_vectors/diagnostics `--standardize` 반영) 선택.

In [ ]:
from google.colab import files
import zipfile, os, subprocess
print('steering_crossmodel_bundle.zip 를 선택하세요 ...')
up = files.upload()
name = next(iter(up))
os.makedirs('/content/asteer', exist_ok=True)
with zipfile.ZipFile(name) as z: z.extractall('/content/asteer')
%cd /content/asteer
print(subprocess.run(['find','.','-maxdepth','2','-type','f'], capture_output=True, text=True).stdout)

## 3. 모델 설정 (여기만 바꾸면 됨)

`build`: Qwen만 `--standardize`(거대활성 다운웨이트). `layer`: Gemma 20 고정, 나머지 auto.
진단 결과 Qwen에 더 나은 층이 보이면 `layer`를 그 값으로 교체.

In [ ]:
import os
os.environ['STEER_4BIT'] = '1'                       # T4:'1' / L4·A100:'0'(추출 품질↑)
COEFFS = "-1,-0.5,-0.25,0,0.25,0.5,1"                # ±2 제거(과주입 회피)
MODELS = [
    {'slug':'gemma9b',   'id':'google/gemma-2-9b-it',               'layer':'20', 'build':''},
    {'slug':'mistral7b', 'id':'mistralai/Mistral-7B-Instruct-v0.3', 'layer':None, 'build':''},
    {'slug':'qwen7b',    'id':'Qwen/Qwen2.5-7B-Instruct',           'layer':None, 'build':'--standardize'},
]
print('STEER_4BIT =', os.environ['STEER_4BIT'], '| coeffs =', COEFFS)
for m in MODELS: print(' -', m['id'], '| layer', m['layer'] or 'auto', '| build', m['build'] or '(raw)')

## 4. 스모크 (extract `--limit 32` → build → steer `--smoke`) — 크래시/경로 확인

In [ ]:
for m in MODELS:
    lay = f"--layer {m['layer']}" if m['layer'] else ""
    print(f"\n########## SMOKE {m['id']} ##########")
    !python steering/extract_activations.py --model {m['id']} --limit 32
    !python steering/build_vectors.py {lay} {m['build']}
    !python steering/steer_eval.py --model {m['id']} {lay} --coeffs={COEFFS} --smoke --out artifacts/vectors/steer_{m['slug']}_smoke.json

## 5. 전체 실행 (extract → **진단** → build → steer_eval)

진단 출력에서 볼 것: `cos(V1,V2)`(≈1 = 일관, 낮으면 잡음), `거대활성 top1%`(높으면 소수 차원 지배 → 표준화 필요), 층별 추이.

In [ ]:
for m in MODELS:
    lay = f"--layer {m['layer']}" if m['layer'] else ""
    print(f"\n########## FULL {m['id']} ##########")
    !python steering/extract_activations.py --model {m['id']}
    print("--- 진단(층스윕: cos_V1_V2 · 거대활성) ---")
    !python steering/diagnostics.py {m['build']} --out artifacts/vectors/diag_{m['slug']}.json
    !python steering/build_vectors.py {lay} {m['build']}
    !python steering/steer_eval.py --model {m['id']} {lay} --coeffs={COEFFS} --out artifacts/vectors/steer_{m['slug']}.json

## 6. 교차모델 비교표 + 다운로드

In [ ]:
import json, os
from google.colab import files
hdr = f"{'model':10} {'L':>3} {'R':>7} {'cos_cue':>7} {'bproj(min->max)':>16} {'dE_ft':>7}  verdict"
print(hdr); print('-' * len(hdr))
for m in MODELS:
    p = f"artifacts/vectors/steer_{m['slug']}.json"
    try:
        r = json.load(open(p))
    except FileNotFoundError:
        print(f"{m['slug']:10}  (missing {p})"); continue
    bp = [s['behavior_proj'] for s in r['sweep']]
    sr = r['summary']['self_report'].get('first_token_isolated', {})
    dE = sr.get('dE')
    cue = r['summary'].get('cue', {}).get('cos_V_cue')
    if dE is None:
        print(f"{m['slug']:10}  (no self_report)"); continue
    rng = max(bp) - min(bp)
    if cue is not None and abs(cue) < 0.1:
        verdict = 'INVALID vector (cos_cue<0.1)'
    elif rng > 2 and abs(dE) < 0.5:
        verdict = 'REPRODUCED (behav O / self X)'
    elif abs(dE) >= 0.5:
        verdict = 'injection moves digit too'
    else:
        verdict = 'inconclusive'
    cs = f"{cue:.3f}" if cue is not None else "  -  "
    print(f"{m['slug']:10} {r['layer']:>3} {r['R']:>7.1f} {cs:>7} {min(bp):+.1f}->{max(bp):+.1f}".ljust(52) +
          f" {dE:>7.3f}  {verdict}")
print('\n판정 규칙: cos_cue<0.1 → 벡터불량(다른 값 무의미). 아니면 behavior 범위 크고 dE_ft≈0 → 착시 재현.')
print('⚠️ behavior_proj 절대값은 R 스케일 달라 모델간 비교 금지 — 샘플 + cos_cue 로 판단.')
for m in MODELS:
    for tag in ('steer', 'diag'):
        p = f"artifacts/vectors/{tag}_{m['slug']}.json"
        if os.path.exists(p): files.download(p)

---
### 해석 가이드
- **cos_cue(=cos_V_cue) 게이트 먼저**: <0.1이면 v_behavior가 외향축을 못 잡은 것 → dE·behavior 다 무의미. (Qwen 1차 실패: 0.018.)
- 게이트 통과 후 **착시 재현** = behavior_proj 범위 큼(+ 샘플이 내향↔외향) & `dE_ft(iso)`≈0.
- **표준화 효과 확인**: Qwen `cos_cue`가 0.018 → 유의미(>~0.2)로 오르고 생성이 코헤런트하면 수리 성공.

### 안 되면(에스컬레이션)
- 진단서 **다른 층**의 cos_V1_V2가 높으면 그 층으로. 여전히 낮으면 col0(싱크) 제외 / LDA·공분산 화이트닝.
- (권장) behavior_proj → **생성텍스트 LLM 심판 채점**으로 행동측정 업그레이드.

### 범위
- behavior 팔만. 완전 이중해리(v_selfreport)는 Phase 1 별도.